# OPV Multi-Objective Bayesian Optimization Tradeoff

This notebook demonstrates a finite-pool multi-objective BO workflow with `matgpr`. The retrospective OPV candidate pool is ranked for a simple materials-design tradeoff: maximize power conversion efficiency while minimizing exciton binding energy.

The notebook uses independent GP surrogates and expected hypervolume improvement.

## 1. Setup

In [ ]:
from __future__ import annotations

import os
import sys
import tempfile
from pathlib import Path

cache_root = Path(tempfile.gettempdir()) / "matgpr_notebook_cache"
os.environ.setdefault("MPLCONFIGDIR", str(cache_root / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(cache_root / "xdg"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "matgpr").exists():
            return candidate
        sibling = candidate / "matgpr"
        if (sibling / "pyproject.toml").exists() and (sibling / "matgpr").exists():
            return sibling
    raise RuntimeError("Could not find the matgpr project root")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from matgpr import (
    ObjectiveSpec,
    select_pareto_front,
    suggest_multi_objective_next_experiments,
    summarize_bo_recommendation_audit,
)

RANDOM_STATE = 44
MEASURED_COUNT = 36
CANDIDATE_COUNT = 100
TOP_K = 8
BO_FIT_MODEL = True
TARGET_COLUMN = "PCE"
BINDING_COLUMN = "E_bind"

BASE_FEATURE_COLUMNS = [
    "polarizability",
    "delLA",
    "delLD",
    "N_atom",
    "Eg",
    "lamda_h",
    "DIP",
    "AL-DH",
    "delHD",
    "E_bind",
    "DL-AL",
    "delGE",
    "E_T1",
]
PHYSICS_COLUMNS = ["physics_degeneracy_score", "physics_binding_score"]

plt.rcParams.update({
    "figure.dpi": 140,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 2. Load OPV Data And Build Features

The notebook uses historical OPV rows as a retrospective finite candidate pool. In a real multi-objective campaign, the candidate table would not include target values.

In [ ]:
def load_opv_data() -> pd.DataFrame:
    data_path = PROJECT_ROOT / "examples" / "opv" / "dataset.pkl"
    data = pd.read_pickle(data_path)
    data = data.rename(columns={"#Sno.": "candidate_id"})
    data["candidate_id"] = "opv_" + data["candidate_id"].astype(str)
    keep_columns = ["candidate_id", TARGET_COLUMN, *BASE_FEATURE_COLUMNS]
    return data.loc[:, keep_columns].dropna().reset_index(drop=True)


def add_physics_scores(frame: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    result = frame.loc[:, BASE_FEATURE_COLUMNS].astype(float).copy()
    physics_columns = ["delHD", "delLD", "delLA", "E_bind"]
    physics_source = frame.loc[:, physics_columns].astype(float)
    reference_physics = reference.loc[:, physics_columns].astype(float)
    reference_mean = reference_physics.mean(axis=0)
    reference_std = reference_physics.std(axis=0, ddof=0).replace(0.0, 1.0)
    z_scores = (physics_source - reference_mean) / reference_std
    result["physics_degeneracy_score"] = -(
        z_scores["delHD"] + z_scores["delLD"] + z_scores["delLA"]
    ) / 3.0
    result["physics_binding_score"] = -z_scores["E_bind"]
    return result


opv_data = load_opv_data()
measured_data = opv_data.sample(n=MEASURED_COUNT, random_state=RANDOM_STATE).reset_index(drop=True)
candidate_pool = opv_data.loc[~opv_data["candidate_id"].isin(measured_data["candidate_id"])]
candidate_pool = candidate_pool.sample(n=CANDIDATE_COUNT, random_state=RANDOM_STATE + 1).reset_index(drop=True)

X_train_raw = add_physics_scores(measured_data, measured_data)
X_candidates_raw = add_physics_scores(candidate_pool, measured_data)
scaler = StandardScaler().fit(X_train_raw)
X_train = pd.DataFrame(scaler.transform(X_train_raw), columns=X_train_raw.columns)
X_candidates = pd.DataFrame(scaler.transform(X_candidates_raw), columns=X_candidates_raw.columns)

candidate_metadata = candidate_pool.loc[:, ["candidate_id", TARGET_COLUMN, BINDING_COLUMN, "Eg", "delHD", "delLD", "delLA"]].copy()
candidate_metadata = candidate_metadata.rename(columns={TARGET_COLUMN: "withheld_pce_for_retrospective"})
candidate_metadata = pd.concat(
    [candidate_metadata.reset_index(drop=True), X_candidates_raw.loc[:, PHYSICS_COLUMNS].reset_index(drop=True)],
    axis=1,
)

print(f"Measured rows: {len(measured_data)}")
print(f"Candidate rows: {len(candidate_pool)}")
candidate_metadata.head()

## 3. Rank The Multi-Objective Candidate Pool

The two objectives are intentionally simple for demonstration:

- maximize OPV PCE,
- minimize exciton binding energy.

BoTorch uses the measured training rows to model both objectives. Withheld PCE values are kept only for retrospective visualization and never feed the ranking.

In [ ]:
retrospective_objectives = [
    ObjectiveSpec(name="PCE", column="withheld_pce_for_retrospective", goal="maximize", weight=0.7),
    ObjectiveSpec(name="binding", column=BINDING_COLUMN, goal="minimize", weight=0.3),
]
y_train = measured_data.loc[:, [TARGET_COLUMN, BINDING_COLUMN]].copy()
bo_result = suggest_multi_objective_next_experiments(
    X_train=X_train,
    y_train=y_train,
    X_candidates=X_candidates,
    objective_directions=("maximize", "minimize"),
    objective_names=("PCE", "binding"),
    candidate_data=candidate_metadata,
    top_k=TOP_K,
    acquisition_function="q_log_expected_hypervolume_improvement",
    normalize_features=True,
    standardize_targets=True,
    fit_model=BO_FIT_MODEL,
    mc_samples=64,
    sampler_seed=RANDOM_STATE,
)
ranked_candidates = bo_result.ranked_candidates
recommendations = bo_result.recommendations
recommendation_audit = summarize_bo_recommendation_audit(
    bo_result,
    candidate_count=len(candidate_pool),
    identifier_columns=("candidate_id",),
)

display(recommendations.head(TOP_K))

## 4. Audit The Recommendations

In [ ]:
display(recommendation_audit.overview_frame())
display(recommendation_audit.policy_summary_frame())
display(recommendation_audit.score_summary_frame())
display(recommendation_audit.recommendation_frame())

## 5. Pareto-Front Visualization

The plot shows the tradeoff between PCE and binding energy in original OPV units. The recommended candidates should lie near favorable high-PCE, low-binding regions or near the predicted Pareto front.

In [ ]:
pareto_candidates = select_pareto_front(candidate_metadata, retrospective_objectives)

fig, ax = plt.subplots(figsize=(6.2, 4.8))
ax.scatter(
    candidate_metadata[BINDING_COLUMN],
    candidate_metadata["withheld_pce_for_retrospective"],
    s=24,
    color="#8a8f98",
    alpha=0.55,
    label="candidate pool",
)
ax.scatter(
    pareto_candidates[BINDING_COLUMN],
    pareto_candidates["withheld_pce_for_retrospective"],
    s=42,
    color="#4a7c59",
    alpha=0.85,
    label="retrospective Pareto front",
)
ax.scatter(
    recommendations[BINDING_COLUMN],
    recommendations["withheld_pce_for_retrospective"],
    s=70,
    color="#007c89",
    edgecolor="black",
    linewidth=0.5,
    label="recommended",
)
for _, row in recommendations.head(5).iterrows():
    ax.annotate(str(row["candidate_id"]), (row[BINDING_COLUMN], row["withheld_pce_for_retrospective"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("Exciton binding energy")
ax.set_ylabel("Withheld OPV PCE (%)")
ax.set_title("Multi-objective OPV tradeoff")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 6. Takeaways

Multi-objective BO helps when the best next experiment is not defined by a single property. In materials design, this pattern naturally extends to maximizing performance while minimizing cost, toxicity, degradation, processing difficulty, or synthesis risk.